Document owner: Shreya Chandra

In [1]:
# Importing required libraries
import pandas as pd
import requests
import random
import statsmodels.api as sm
from datetime import timedelta
from tqdm import tqdm
import time
import os
import nltk
from IPython.display import display
import numpy as np
tqdm.pandas()
import warnings
warnings.filterwarnings('ignore')
import re, io
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter, Retry
try:
  import lxml
except:
  !pip install lxml
  import lxml
from io import StringIO
try:
  import xlsxwriter
except:
  !pip install xlsxwriter
  import xlsxwriter
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.3/172.3 kB 3.5 MB/s eta 0:00:00


In [8]:
# scraping Wikipedia site
r = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers={'User-agent': 'Chrome/103.0.5060.114'}).text
df = pd.read_html(StringIO(str(r))) #load with user agent to avoid 401 error
CIK_symbols = pd.DataFrame(df[0])

In [9]:
CIK_symbols['CIK'] = CIK_symbols['CIK'].astype(str).str.zfill(10)

In [ ]:
print(CIK_symbols[['Symbol', 'CIK']])

    Symbol         CIK
0      MMM  0000066740
1      AOS  0000091142
2      ABT  0000001800
3     ABBV  0001551152
4      ACN  0001467373
..     ...         ...
498    XYL  0001524472
499    YUM  0001041061
500   ZBRA  0000877212
501    ZBH  0001136869
502    ZTS  0001555280

[503 rows x 2 columns]


In [ ]:
# debugging. Please ignore.
# CIK_symbols = CIK_symbols[:5]

In [ ]:
print(CIK_symbols)

    Symbol             Security             GICS Sector  \
0      MMM                   3M             Industrials   
1      AOS          A. O. Smith             Industrials   
2      ABT  Abbott Laboratories             Health Care   
3     ABBV               AbbVie             Health Care   
4      ACN            Accenture  Information Technology   
..     ...                  ...                     ...   
498    XYL           Xylem Inc.             Industrials   
499    YUM          Yum! Brands  Consumer Discretionary   
500   ZBRA   Zebra Technologies  Information Technology   
501    ZBH        Zimmer Biomet             Health Care   
502    ZTS               Zoetis             Health Care   

                                GICS Sub-Industry    Headquarters Location  \
0                        Industrial Conglomerates    Saint Paul, Minnesota   
1                               Building Products     Milwaukee, Wisconsin   
2                           Health Care Equipment  North 

In [21]:
# extracting master files for 2024 and 2025 - all quarters, from sec
# Example, the master.idx file for 2024 and QTR1 is below-
# https://www.sec.gov/Archives/edgar/full-index/2024/QTR1/
HEADERS = {
    "User-Agent": "research/education (shrchandra@ucdavis.edu)",
    "Accept-Encoding": "gzip, deflate",
}


# --- helpers ---
def make_session():
    s = requests.Session()
    retries = Retry(total=5, backoff_factor=0.6,
                    status_forcelist=(429, 500, 502, 503, 504),
                    allowed_methods=frozenset(["GET"]))
    s.mount("https://", HTTPAdapter(max_retries=retries))
    return s

SESSION = make_session()

year = 2020
qtr = "QTR1"
if year == 2020 and qtr == "QTR1":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2020_QTR1 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2020
qtr = "QTR2"
if year == 2020 and qtr == "QTR2":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2020_QTR2 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2020
qtr = "QTR3"
if year == 2020 and qtr == "QTR3":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2020_QTR3 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")
year = 2020
qtr = "QTR4"
if year == 2020 and qtr == "QTR4":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2020_QTR4 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2021
qtr = "QTR1"
if year == 2021 and qtr == "QTR1":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2021_QTR1 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2021
qtr = "QTR2"
if year == 2021 and qtr == "QTR2":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2021_QTR2 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2021
qtr = "QTR3"
if year == 2021 and qtr == "QTR3":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2021_QTR3 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2021
qtr = "QTR4"
if year == 2021 and qtr == "QTR4":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2021_QTR4 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2022
qtr = "QTR1"
if year == 2022 and qtr == "QTR1":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2022_QTR1 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2022
qtr = "QTR2"
if year == 2022 and qtr == "QTR2":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2022_QTR2 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2022
qtr = "QTR3"
if year == 2022 and qtr == "QTR3":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2022_QTR3 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2022
qtr = "QTR4"
if year == 2022 and qtr == "QTR4":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2022_QTR4 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2023
qtr = "QTR1"
if year == 2023 and qtr == "QTR1":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2023_QTR1 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2023
qtr = "QTR2"
if year == 2023 and qtr == "QTR2":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2023_QTR2 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2023
qtr = "QTR3"
if year == 2023 and qtr == "QTR3":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2023_QTR3 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2023
qtr = "QTR4"
if year == 2023 and qtr == "QTR4":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2023_QTR4 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")


year = 2024
qtr = "QTR1"
if year == 2024 and qtr == "QTR1":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2024_QTR1 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2024
qtr = "QTR2"
if year == 2024 and qtr == "QTR2":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2024_QTR2 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2024
qtr = "QTR3"
if year == 2024 and qtr == "QTR3":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2024_QTR3 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2024
qtr = "QTR4"
if year == 2024 and qtr == "QTR4":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2024_QTR4 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2025
qtr = "QTR1"
if year == 2025 and qtr == "QTR1":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2025_QTR1 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2025
qtr = "QTR2"
if year == 2025 and qtr == "QTR2":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2025_QTR2 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2025
qtr = "QTR3"
if year == 2025 and qtr == "QTR3":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2025_QTR3 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

year = 2025
qtr = "QTR4"
if year == 2025 and qtr == "QTR4":
  url = f"https://www.sec.gov/Archives/edgar/full-index/{year}/{qtr}/master.idx"
  r = SESSION.get(url, headers=HEADERS, timeout=30)
  r.raise_for_status()
  idx_2025_QTR4 = pd.read_table(io.BytesIO(r.content), sep="|", skiprows=11,
                        names=["cik","company_name","form_type","date_filed","txt_filename"],
                        encoding="latin-1")

In [ ]:
# run below code to run for entire list of CKIDs
pd.set_option('display.max_colwidth', None)

def filing_index_url(txt_filename: str) -> str:
    # folder: edgar/data/320193/000032019324000080
    folder = "/".join(txt_filename.split("/")[:-1])
    # base from the .txt (e.g., 0000320193-24-000080)
    base = os.path.splitext(os.path.basename(txt_filename))[0]
    # try .htm then .html
    return f"https://www.sec.gov/Archives/{folder}/{base}-index.htm"

def try_alt_index_url(url_htm: str) -> str:
    if url_htm.endswith(".htm"):
        url_html = url_htm + "l"    # .html
        # quick HEAD/GET check
        resp = SESSION.get(url_htm, headers=HEADERS, timeout=15)
        if resp.status_code == 200:
            return url_htm
        resp2 = SESSION.get(url_html, headers=HEADERS, timeout=15)
        if resp2.status_code == 200:
            return url_html
        # last resort: list directory and find *-index.htm*
        directory = url_htm.rsplit("/", 1)[0] + "/"
        resp3 = SESSION.get(directory, headers=HEADERS, timeout=15)
        if resp3.ok:
            m = re.search(r'href="([^"]+-index\.html?)"', resp3.text, flags=re.I)
            if m:
                return requests.compat.urljoin(directory, m.group(1))
    return url_htm  # fallback

def polite_pause():
    time.sleep(0.5 + random.random()*0.7)  # stay nice to SEC

def find_ex99_1_url(index_url):
    index_url = try_alt_index_url(index_url)
    r = SESSION.get(index_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    # Primary: row labeled EX-99.1
    for row in soup.select("table tr"):
        tds = row.find_all("td")
        if not tds:
            continue
        text = " ".join(td.get_text(" ", strip=True) for td in tds).upper()
        Ex99regex = ["EX-99.1", "EX-99"]
        if any(Ex99 in text for Ex99 in Ex99regex):
            link = row.find("a", href=True)
            if link:
                return requests.compat.urljoin(index_url, link["href"])
    # Fallback: any href that looks like ex99.1
    for a in soup.find_all("a", href=True):
        if re.search(r"ex-?99\.1|ex99\.1|ex991", a["href"], flags=re.I):
            return requests.compat.urljoin(index_url, a["href"])
    return None

DATE_RE = re.compile(r"\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}\b")

def extract_date_from_exhibit(ex99_url):
    if not ex99_url:
        return None
    polite_pause()
    r = SESSION.get(ex99_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    text = BeautifulSoup(r.text, "html.parser").get_text(" ", strip=True)
    # Heuristic: capture the first explicit "Month Day, Year", worked all times.
    # Checking manually for each company if it worked or not
    # preferring one near "today announced" or "announced"
    ann = re.search(r"(?:today\s+)?announced.*?\b(" + DATE_RE.pattern + ")", text, flags=re.I)
    if ann:
        return ann.group(1)
    m = DATE_RE.search(text)
    return m.group(0) if m else None

def includes_item_202(index_url: str) -> bool:
    index_url = try_alt_index_url(index_url)
    polite_pause()
    r = SESSION.get(index_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    # Search the index page text for "Item 2.02"
    # Item 2.02 indicates:
    # Item 2.02 of Form 8-K is specifically titled “Results of Operations and Financial Condition.”
    # Companies use it when they furnish earnings releases or other announcements about quarterly or annual financial results.
    # The SEC even clarifies that if the 8-K includes information under Item 2.02, it’s treated as a furnishing of earnings results, not a filing of them, to limit certain liabilities.
    text = BeautifulSoup(r.text, "html.parser").get_text(" ", strip=True)
    return bool(re.search(r"\bItem\s*2\.02\b", text, flags=re.I))


# --- main: Company 8-Ks in 2024–2025 ---
def get_earnings_date(TICKR_CIK, TICKR_SYMBOL):
    rows = []
    for y in [2024, 2025]:
        for q in ["QTR1","QTR2","QTR3","QTR4"]:
            idx = f"idx_{y}_{q}"
            idx = globals()[idx]
            # try:
            #     idx = get_master_idx(y, q)
            # except requests.HTTPError:
            #     continue
            sub = idx[(idx["form_type"]=="8-K") & (idx["cik"].astype(str).str.zfill(10)==TICKR_CIK)]
            if not sub.empty:
              for _, r in sub.iterrows():
                  polite_pause()
                  idx_url = filing_index_url(r["txt_filename"])
                  # print(f"idx_url: {idx_url}")
                  try:
                      if not includes_item_202(idx_url):
                          continue
                  except requests.HTTPError:
                      # if index page can’t be fetched, skip this filing
                      continue
                  try:
                      ex99_url = find_ex99_1_url(idx_url)
                  except requests.HTTPError:
                      # skip problematic filings gracefully
                      ex99_url = None
                  # print(f"ex99_url: {ex99_url}")
                  ex_date = extract_date_from_exhibit(ex99_url) if ex99_url else None
                  if ex_date:
                    rows.append({
                        "cik": r["cik"],
                        "company": r["company_name"],
                        "form_type": r["form_type"],
                        "filing_date": r["date_filed"],     # reliable: announcement date
                        "index_url": idx_url,
                        "ex99_url": ex99_url,
                        "exhibit_date_found": ex_date
                    })
                  else: continue
            else:
                print(f"No SEC filing found for- {TICKR_CIK} for {y}-{q}")
                continue

    if rows == []:
      earnings_df = pd.DataFrame({'returnStatus': [f"No SEC filing found for- {TICKR_SYMBOL}"]})
    else:
      earnings_df = pd.DataFrame(rows).sort_values("filing_date").reset_index(drop=True)
    print(earnings_df)
    return earnings_df
# get_earnings_date("0001467373")
# with pd.ExcelWriter("earnings_data.xlsx", engine="openpyxl") as writer:
#   for index, each_row in CIK_symbols.iterrows():
#       earnings_df = get_earnings_date(each_row['CIK'])
#       sheet_name = str(each_row['CIK'])
#       earnings_df.to_excel(writer, sheet_name=sheet_name, index=False)

out_file = Path("earnings_data.xlsx")
first_write = True
for index, each_row in CIK_symbols.iterrows():
  sheet_name = str(each_row['Symbol'])
  cik = str(each_row["CIK"])
  try:
    earnings_df = get_earnings_date(each_row['CIK'], each_row['Symbol'])
    mode = "w" if first_write else "a"
    if mode == "w":
      with pd.ExcelWriter(out_file, engine="openpyxl", mode=mode) as writer:
        earnings_df.to_excel(writer, sheet_name=sheet_name, index=False)
    else:
      with pd.ExcelWriter(out_file, engine="openpyxl", mode=mode, if_sheet_exists="new") as writer:
        earnings_df.to_excel(writer, sheet_name=sheet_name, index=False)
    first_write = False
  except Exception as e:
    print(f"Error for CIK {cik}: {e}")

Streaming output truncated to the last 5000 lines.
2  https://www.sec.gov/Archives/edgar/data/798354/0000798354-24-000098-index.htm   
3  https://www.sec.gov/Archives/edgar/data/798354/0000798354-24-000154-index.htm   
4  https://www.sec.gov/Archives/edgar/data/798354/0000798354-24-000188-index.htm   
5  https://www.sec.gov/Archives/edgar/data/798354/0000798354-25-000018-index.htm   
6  https://www.sec.gov/Archives/edgar/data/798354/0000798354-25-000102-index.htm   
7  https://www.sec.gov/Archives/edgar/data/798354/0000798354-25-000161-index.htm   

                                                                                      ex99_url  \
0  https://www.sec.gov/Archives/edgar/data/798354/000079835424000021/fi4q23earningsrelease.htm   
1          https://www.sec.gov/Archives/edgar/data/798354/000119312524077703/d817941dex991.htm   
2  https://www.sec.gov/Archives/edgar/data/798354/000079835424000098/fiq124earningsrelease.htm   
3  https://www.sec.gov/Archives/edgar/data/798354/00

KeyboardInterrupt: 

Pros:   
1. This method is more robust. We can scale it back in time (before 2024 also).   
2. There is no dependence on external aggregator website like nasdaq/yfinance. Correctness of data can be trusted more as all companies are mandated to file to sec.   
3. In future, we can extract more information from the sec filings to enrich our research work. Companies report many documents to sec. One idea is to run a sentiment analysis on 8-K filings itself and use it as an input for next month's contrarian identification.   
4. Since the initial steps in this pipeline involes hitting sec API and extracting data, those steps are streamlined.  
***********************************
Cons:     
4. Slight disrepancies were observed in the Excel generated from the nasdaq site. This happens in the function extract_date_from_exhibit, when we extract earnings dates from the Ex 99.1 page. However those were minor - some duplicate dates were printed in Excel, lag or lead of one day from date on nasdaq site. For now, each company's nasdaq website was visited manually and corrections were made in the Excel to reflect information on nasdaq website.  
5. This code took ~6 hours to run (for 500 companies).   
6. Since this data now has to be passed to the stocknewsapi, we will have to find a novel way to parse this data for the API.


In [15]:
CIK_symbols = CIK_symbols[(CIK_symbols['Symbol'] == "FOX") | (CIK_symbols['Symbol'] == "NWS") | (CIK_symbols['Symbol'] == "PSKY")]
print(CIK_symbols)

    Symbol                        Security             GICS Sector  \
203    FOX       Fox Corporation (Class B)  Communication Services   
335    NWS             News Corp (Class B)  Communication Services   
361   PSKY  Paramount Skydance Corporation  Communication Services   

          GICS Sub-Industry    Headquarters Location  Date added         CIK  \
203            Broadcasting  New York City, New York  2019-03-19  0001754301   
335              Publishing  New York City, New York  2015-09-18  0001564708   
361  Movies & Entertainment  Los Angeles, California  1994-09-30  0002041610   

                            Founded  
203                            2019  
335    2013 (News Corporation 1980)  
361  2025 (Paramount Pictures 1912)  


In [22]:
# run below code to run for entire list of CKIDs
pd.set_option('display.max_colwidth', None)

def filing_index_url(txt_filename: str) -> str:
    # folder: edgar/data/320193/000032019324000080
    folder = "/".join(txt_filename.split("/")[:-1])
    # base from the .txt (e.g., 0000320193-24-000080)
    base = os.path.splitext(os.path.basename(txt_filename))[0]
    # try .htm then .html
    return f"https://www.sec.gov/Archives/{folder}/{base}-index.htm"

def try_alt_index_url(url_htm: str) -> str:
    if url_htm.endswith(".htm"):
        url_html = url_htm + "l"    # .html
        # quick HEAD/GET check
        resp = SESSION.get(url_htm, headers=HEADERS, timeout=15)
        if resp.status_code == 200:
            return url_htm
        resp2 = SESSION.get(url_html, headers=HEADERS, timeout=15)
        if resp2.status_code == 200:
            return url_html
        # last resort: list directory and find *-index.htm*
        directory = url_htm.rsplit("/", 1)[0] + "/"
        resp3 = SESSION.get(directory, headers=HEADERS, timeout=15)
        if resp3.ok:
            m = re.search(r'href="([^"]+-index\.html?)"', resp3.text, flags=re.I)
            if m:
                return requests.compat.urljoin(directory, m.group(1))
    return url_htm  # fallback

def polite_pause():
    time.sleep(0.5 + random.random()*0.7)  # stay nice to SEC

def find_ex99_1_url(index_url):
    index_url = try_alt_index_url(index_url)
    r = SESSION.get(index_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    # Primary: row labeled EX-99.1
    for row in soup.select("table tr"):
        tds = row.find_all("td")
        if not tds:
            continue
        text = " ".join(td.get_text(" ", strip=True) for td in tds).upper()
        Ex99regex = ["EX-99.1", "EX-99"]
        if any(Ex99 in text for Ex99 in Ex99regex):
            link = row.find("a", href=True)
            if link:
                return requests.compat.urljoin(index_url, link["href"])
    # Fallback: any href that looks like ex99.1
    for a in soup.find_all("a", href=True):
        if re.search(r"ex-?99\.1|ex99\.1|ex991", a["href"], flags=re.I):
            return requests.compat.urljoin(index_url, a["href"])
    return None

DATE_RE = re.compile(r"\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}\b")

def extract_date_from_exhibit(ex99_url):
    if not ex99_url:
        return None
    polite_pause()
    r = SESSION.get(ex99_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    text = BeautifulSoup(r.text, "html.parser").get_text(" ", strip=True)
    # Heuristic: capture the first explicit "Month Day, Year", worked all times.
    # Checking manually for each company if it worked or not
    # preferring one near "today announced" or "announced"
    ann = re.search(r"(?:today\s+)?announced.*?\b(" + DATE_RE.pattern + ")", text, flags=re.I)
    if ann:
        return ann.group(1)
    m = DATE_RE.search(text)
    return m.group(0) if m else None

def includes_item_202(index_url: str) -> bool:
    index_url = try_alt_index_url(index_url)
    polite_pause()
    r = SESSION.get(index_url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    # Search the index page text for "Item 2.02"
    # Item 2.02 indicates:
    # Item 2.02 of Form 8-K is specifically titled “Results of Operations and Financial Condition.”
    # Companies use it when they furnish earnings releases or other announcements about quarterly or annual financial results.
    # The SEC even clarifies that if the 8-K includes information under Item 2.02, it’s treated as a furnishing of earnings results, not a filing of them, to limit certain liabilities.
    text = BeautifulSoup(r.text, "html.parser").get_text(" ", strip=True)
    return bool(re.search(r"\bItem\s*2\.02\b", text, flags=re.I))


# --- main: Company 8-Ks in 2024–2025 ---
def get_earnings_date(TICKR_CIK, TICKR_SYMBOL):
    rows = []
    for y in [2020, 2021, 2022, 2023, 2024]:
        for q in ["QTR1","QTR2","QTR3","QTR4"]:
            idx = f"idx_{y}_{q}"
            idx = globals()[idx]
            # try:
            #     idx = get_master_idx(y, q)
            # except requests.HTTPError:
            #     continue
            sub = idx[(idx["form_type"]=="8-K") & (idx["cik"].astype(str).str.zfill(10)==TICKR_CIK)]
            if not sub.empty:
              for _, r in sub.iterrows():
                  polite_pause()
                  idx_url = filing_index_url(r["txt_filename"])
                  # print(f"idx_url: {idx_url}")
                  try:
                      if not includes_item_202(idx_url):
                          continue
                  except requests.HTTPError:
                      # if index page can’t be fetched, skip this filing
                      continue
                  try:
                      ex99_url = find_ex99_1_url(idx_url)
                  except requests.HTTPError:
                      # skip problematic filings gracefully
                      ex99_url = None
                  # print(f"ex99_url: {ex99_url}")
                  ex_date = extract_date_from_exhibit(ex99_url) if ex99_url else None
                  if ex_date:
                    rows.append({
                        "cik": r["cik"],
                        "company": r["company_name"],
                        "form_type": r["form_type"],
                        "filing_date": r["date_filed"],     # reliable: announcement date
                        "index_url": idx_url,
                        "ex99_url": ex99_url,
                        "exhibit_date_found": ex_date
                    })
                  else: continue
            else:
                print(f"No SEC filing found for- {TICKR_CIK} for {y}-{q}")
                continue

    if rows == []:
      earnings_df = pd.DataFrame({'returnStatus': [f"No SEC filing found for- {TICKR_SYMBOL}"]})
    else:
      earnings_df = pd.DataFrame(rows).sort_values("filing_date").reset_index(drop=True)
    print(earnings_df)
    return earnings_df
# get_earnings_date("0001467373")
# with pd.ExcelWriter("earnings_data.xlsx", engine="openpyxl") as writer:
#   for index, each_row in CIK_symbols.iterrows():
#       earnings_df = get_earnings_date(each_row['CIK'])
#       sheet_name = str(each_row['CIK'])
#       earnings_df.to_excel(writer, sheet_name=sheet_name, index=False)

out_file = Path("earnings_dataVf_3companies.xlsx")
first_write = True
for index, each_row in CIK_symbols.iterrows():
  sheet_name = str(each_row['Symbol'])
  cik = str(each_row["CIK"])
  try:
    earnings_df = get_earnings_date(each_row['CIK'], each_row['Symbol'])
    mode = "w" if first_write else "a"
    if mode == "w":
      with pd.ExcelWriter(out_file, engine="openpyxl", mode=mode) as writer:
        earnings_df.to_excel(writer, sheet_name=sheet_name, index=False)
    else:
      with pd.ExcelWriter(out_file, engine="openpyxl", mode=mode, if_sheet_exists="new") as writer:
        earnings_df.to_excel(writer, sheet_name=sheet_name, index=False)
    first_write = False
  except Exception as e:
    print(f"Error for CIK {cik}: {e}")

        cik   company form_type filing_date  \
0   1754301  Fox Corp       8-K  2020-02-05   
1   1754301  Fox Corp       8-K  2020-05-06   
2   1754301  Fox Corp       8-K  2020-08-04   
3   1754301  Fox Corp       8-K  2020-11-03   
4   1754301  Fox Corp       8-K  2021-02-09   
5   1754301  Fox Corp       8-K  2021-05-05   
6   1754301  Fox Corp       8-K  2021-08-04   
7   1754301  Fox Corp       8-K  2021-11-03   
8   1754301  Fox Corp       8-K  2022-02-09   
9   1754301  Fox Corp       8-K  2022-05-10   
10  1754301  Fox Corp       8-K  2022-08-10   
11  1754301  Fox Corp       8-K  2022-11-01   
12  1754301  Fox Corp       8-K  2023-02-08   
13  1754301  Fox Corp       8-K  2023-05-09   
14  1754301  Fox Corp       8-K  2023-08-08   
15  1754301  Fox Corp       8-K  2023-11-02   
16  1754301  Fox Corp       8-K  2024-02-07   
17  1754301  Fox Corp       8-K  2024-05-08   
18  1754301  Fox Corp       8-K  2024-08-06   
19  1754301  Fox Corp       8-K  2024-11-04   

            